# 15 — Within $1\,R_{200c}$ vs all satellites: does the radius cut change the quenching-anisotropy fit?

Mirrors `14_tng100_anisotropy_hostmass_comparison_logM8`, but instead of comparing host-mass cuts
alone it adds a **radius** axis: for each host-mass cut (TNG100, z=0 and z=0.05, $M_{*,\rm sat}>10^8\,M_\odot$)
it compares the $f_q(\theta)=a+b\cos 2\theta$ MCMC fit for

* **`<1 R200c`** — satellites within $1\,R_{200c}$ (the cut used everywhere so far), and
* **`all sat`** — every FoF satellite of the host, **no radius cut**.

**Why this notebook rebuilds catalogs.** The catalogs written by notebooks 01 / 08–13 already had
the $<1\,R_{200c}$ cut applied *before* saving, so the beyond-$R_{200c}$ satellites are not on disk.
Part 1 therefore rebuilds **no-radius-cut** catalogs from the raw TNG100 data (keeping the
`d_r200_3d` column); Part 2 recovers the $<1\,R_{200c}$ sample by filtering `d_r200_3d < 1`, so both
fits come from one consistent superset.

Requires the raw TNG100 data under `~/SimulationData/L75n1820TNG` and `illustris_python` (run on the
cluster/Binder, as with notebooks 01 / 08–13).

# Part 1 — rebuild no-radius-cut catalogs from raw TNG100

In [ ]:
import os, glob, re
import numpy as np
import pandas as pd
import h5py
import illustris_python as il

## Configuration

`HOST_CUTS` are the same selections as notebook 14 (now each with an explicit filename tag, including
the MW-mass baseline). We build the `M*_sat > 1e8` catalog with **no radius cut** for each host cut at
both redshifts, writing files tagged `_allsat_` so they never collide with the within-$R_{200c}$
catalogs.

In [ ]:
SIM      = 'TNG100'
h        = 0.6774                          # IllustrisTNG little-h
SN_MIN   = 2.5                             # host SDSS-morphology S/N quality cut
LOGCUT   = 8                               # satellite log10(M*/Msun) cut: M*_sat > 1e8

_SIM_DIR = {'TNG100': 'L75n1820TNG', 'TNG50': 'L35n2160TNG'}[SIM]
_LBOX_MPC_H = {'TNG100': 75.0, 'TNG50': 35.0}[SIM]
TNG_ROOT = os.path.expanduser(f'~/SimulationData/{_SIM_DIR}')
Lbox     = _LBOX_MPC_H * 1000.0 / h        # box side in physical kpc

# redshift -> (subdir, snapshot, z) ; project convention z=0.05 -> snap 95
REDSHIFTS = {
    'z0':    ('z0',    99, 0.0),
    'z0p05': ('z0p05', 95, 0.0485),
}

# host-mass selections: (label, filename tag, logM200c min, logM200c max-or-None)
HOST_CUTS = [
    ("12.0-12.5", "hostlogM12.0-12.5", 12.0, 12.5),   # MW-mass baseline
    (">12",       "hostlogM12.0plus",  12.0, None),
    ("13-14",     "hostlogM13.0-14.0", 13.0, 14.0),
    (">13",       "hostlogM13.0plus",  13.0, None),
]

## Shared build helpers (identical to notebooks 01 / 08–13)

In [ ]:
def wrap_min_image(d, L):
    return d - L * np.round(d / L)

def angle_from_major_axis(phi_sat_deg, pa_host_deg):
    """Fold |phi_sat - PA_host| into [0, 90] using the disk's 180-deg / mirror symmetry."""
    phi = np.mod(phi_sat_deg, 360.0)
    d1 = np.abs(phi - np.mod(pa_host_deg, 360.0))
    d2 = np.abs(phi - np.mod(pa_host_deg + 180.0, 360.0))
    d1 = np.minimum(d1, 360.0 - d1); d2 = np.minimum(d2, 360.0 - d2)
    m  = np.minimum(d1, d2)
    return np.minimum(m, 180.0 - m)

def quenched_flag(logmstar_phys, sfr):
    """1 if >= 1 dex below the SDSS SFMS (Martin-Navarro+ 2021), else 0."""
    sfr_ms = 10.0 ** (0.75 * logmstar_phys - 7.5)
    return (sfr < sfr_ms / 10.0).astype(int)

def host_morph_to_satellites(sdss, first_sub, n_subs, subfind_ids, n_sub):
    """Broadcast each selected host's Sersic morphology to its satellites."""
    theta = np.zeros(n_sub); flag = np.zeros(n_sub); sflag = np.zeros(n_sub); sn = np.zeros(n_sub)
    theta_all = np.asarray(sdss['sersic_theta']) * 180.0 / np.pi
    flag_all  = np.asarray(sdss['flag'])
    sflag_all = np.asarray(sdss['flag_sersic'])
    sn_all    = np.asarray(sdss['sn_per_pixel'])
    for k in range(len(first_sub)):
        c   = first_sub[k]
        row = np.where(subfind_ids == c)[0]
        if row.size == 0:
            flag[c + 1:c + n_subs[k]] = 1
            continue
        r  = row[0]
        sl = slice(c + 1, c + n_subs[k])
        theta[sl] = theta_all[r]; flag[sl] = flag_all[r]
        sflag[sl] = sflag_all[r]; sn[sl] = sn_all[r]
    return theta, flag, sflag, sn

## Build one snapshot, all host cuts (no radius cut)

Loads the group/subhalo/morphology data for a redshift once, then loops the host-mass cuts. For each
cut it flags FoF satellites, broadcasts host orientation + $R_{200c}$, computes the projected angle
`alpha` and `d_r200_3d`, applies only the `M*_sat > 1e8` and reliable-host-morphology cuts (**no
radius cut**), and writes `..._{tag}_allsat_logM8.00.csv`.

In [ ]:
def build_snapshot(redshift):
    zsub, snap, zred = REDSHIFTS[redshift]
    _A   = 1.0 / (1.0 + zred)                       # comoving -> physical
    _sp  = f'snapnum_{snap:03d}'
    basePath        = os.path.join(TNG_ROOT, 'output')
    morph_g         = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/morphs_g.hdf5')
    morph_i         = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/morphs_i.hdf5')
    subfind_id_path = os.path.join(TNG_ROOT, f'postprocessing/skirt_images/sdss/{_sp}/subfind_ids.txt')
    _gc = os.path.join(basePath, f'groups_{snap:03d}', f'fof_subhalo_tab_{snap:03d}.0.hdf5')
    assert os.path.exists(_gc),     f'group catalog not found: {_gc}'
    assert os.path.exists(morph_g), f'SDSS morphology not found: {morph_g}'

    OUTDIR = f'../data/{SIM.lower()}/{zsub}'
    os.makedirs(OUTDIR, exist_ok=True)

    groups = il.groupcat.loadHalos(basePath, snap,
        fields=['GroupFirstSub', 'Group_M_Crit200', 'Group_R_Crit200', 'GroupNsubs'])
    subhalos = il.groupcat.loadSubhalos(basePath, snap,
        fields=['SubhaloGrNr', 'SubhaloMassType', 'SubhaloCM', 'SubhaloSFR'])
    n_sub = len(subhalos['SubhaloGrNr'])
    sdss_g = h5py.File(morph_g, 'r'); sdss_i = h5py.File(morph_i, 'r')
    subfind_ids = np.loadtxt(subfind_id_path)

    with np.errstate(divide='ignore'):
        host_logm200_phys = np.log10(groups['Group_M_Crit200'] * 1e10 / h)
        mstar_phys = np.log10(subhalos['SubhaloMassType'][:, 4] * 1e10 / h)
        mstar_hinv = np.log10(subhalos['SubhaloMassType'][:, 4] * 1e10)
    sat_sfr = np.asarray(subhalos['SubhaloSFR'])
    cm_phys = subhalos['SubhaloCM'] / h

    print(f'\n=== {SIM} {redshift} (snap {snap}, z={zred}) | {n_sub} subhalos ===')
    written = []
    for hlabel, tag, lo, hi in HOST_CUTS:
        host_sel = host_logm200_phys > lo
        if hi is not None:
            host_sel &= host_logm200_phys < hi
        first_sub = groups['GroupFirstSub'][host_sel]
        n_subs    = groups['GroupNsubs'][host_sel]
        r200_all  = groups['Group_R_Crit200'][host_sel] / h
        m200p_all = host_logm200_phys[host_sel]
        m200h_all = np.log10(groups['Group_M_Crit200'][host_sel] * 1e10)

        sat_mask       = np.zeros(n_sub, dtype=bool)
        host_id_arr    = np.full(n_sub, -1, dtype=int)
        host_center    = np.zeros((n_sub, 3))
        host_r200_arr  = np.zeros(n_sub)
        host_m200_phys = np.zeros(n_sub)
        host_m200_hinv = np.zeros(n_sub)
        for k in range(len(first_sub)):
            c  = first_sub[k]; sl = slice(c + 1, c + n_subs[k])
            sat_mask[sl] = True; host_id_arr[sl] = k
            host_center[sl] = cm_phys[c]
            host_r200_arr[sl] = r200_all[k]
            host_m200_phys[sl] = m200p_all[k]; host_m200_hinv[sl] = m200h_all[k]

        tg, fg, sfg, sng = host_morph_to_satellites(sdss_g, first_sub, n_subs, subfind_ids, n_sub)
        ti, fi, sfi, sni = host_morph_to_satellites(sdss_i, first_sub, n_subs, subfind_ids, n_sub)
        host_theta = 0.5 * (tg + ti)
        host_good  = ((fg == 0) & (sfg == 0) & (sng > SN_MIN) &
                      (fi == 0) & (sfi == 0) & (sni > SN_MIN))

        rel    = wrap_min_image(cm_phys - host_center, Lbox)
        phi_2d = np.degrees(np.arctan2(rel[:, 1], rel[:, 0]))
        alpha  = angle_from_major_axis(phi_2d, host_theta)
        _d3d_com = np.linalg.norm(rel, axis=1)
        with np.errstate(invalid='ignore', divide='ignore'):
            d_r200_3d = np.where(host_r200_arr > 0, _d3d_com / host_r200_arr, np.nan)
        d_3d_kpc = _d3d_com * _A

        # NO radius cut: only satellite mass + reliable host morphology
        sel = sat_mask & host_good & (mstar_phys > LOGCUT)
        df = pd.DataFrame({
            'host_id':        host_id_arr[sel],
            'mstar_phys':     mstar_phys[sel],
            'mstar_hinv':     mstar_hinv[sel],
            'host_m200_phys': host_m200_phys[sel],
            'host_m200_hinv': host_m200_hinv[sel],
            'sfr':            sat_sfr[sel],
            'alpha':          alpha[sel],
            'd_3d_kpc':       d_3d_kpc[sel],
            'd_r200_3d':      d_r200_3d[sel],
            'quenched':       quenched_flag(mstar_phys[sel], sat_sfr[sel]),
        })
        assert df.alpha.between(0, 90).all(), 'alpha outside [0, 90]!'
        out = f'{OUTDIR}/tng100_satellites_{tag}_allsat_logM{LOGCUT:.2f}.csv'
        df.to_csv(out, index=False)
        n_in = int((df.d_r200_3d < 1.0).sum())
        written.append(out)
        print(f'  {hlabel:<10s} N_all={len(df):6d}  N(<R200c)={n_in:6d}  '
              f'f_q(all)={df.quenched.mean():.3f}  -> {os.path.basename(out)}')
    return written

## Run the build for both redshifts

In [ ]:
for rs in REDSHIFTS:
    build_snapshot(rs)
print('\ndone -- no-radius-cut catalogs written for both redshifts.')

# Part 2 — compare the MCMC fits: $<1\,R_{200c}$ vs all satellites

Loads the `_allsat_` catalogs, derives the $<1\,R_{200c}$ subset by filtering `d_r200_3d < 1`, and
runs the same $a+b\cos 2\theta$ sinusoid MCMC on both. Same model/plot style as notebooks 03 / 06 / 14.

In [ ]:
import emcee
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
})

DATA_ROOT = "../data"
R200C_FACTOR = 1.0

# radius variants: (label, color, linestyle, marker)
VARIANTS = [
    ("<1 R200c", "#1f77b4", "-",  "o"),    # the cut used everywhere so far
    ("all sat",  "#d62728", "--", "s"),    # no radius cut
]
REDS = [("z=0", "z0"), ("z=0.05", "z0p05")]

# 18 angle bins over [0, 90]
N_BINS        = 18
ANGLE_EDGES   = np.linspace(0, 90, N_BINS + 1)
ANGLE_CENTERS = 0.5 * (ANGLE_EDGES[:-1] + ANGLE_EDGES[1:])

## Load the all-satellite catalogs and split into the two radius variants

In [ ]:
# data[(hlabel, zlabel, vlabel)] -> DataFrame
data = {}
print(f"{'host':<10s} {'redshift':<8s} {'variant':<9s} {'N_sat':>7s} {'f_q':>6s}")
for hlabel, tag, lo, hi in HOST_CUTS:
    for zlabel, zsub in REDS:
        path = os.path.join(DATA_ROOT, SIM.lower(), zsub,
                            f"tng100_satellites_{tag}_allsat_logM{LOGCUT:.2f}.csv")
        if not os.path.exists(path):
            print(f"{hlabel:<10s} {zlabel:<8s} {'(missing)':<9s}  run Part 1 -> {path}")
            continue
        df_all = pd.read_csv(path)
        df_all = df_all[df_all["mstar_phys"] > LOGCUT].reset_index(drop=True)
        df_in  = df_all[df_all["d_r200_3d"] < R200C_FACTOR].reset_index(drop=True)
        for vlabel, df in [("<1 R200c", df_in), ("all sat", df_all)]:
            data[(hlabel, zlabel, vlabel)] = df
            print(f"{hlabel:<10s} {zlabel:<8s} {vlabel:<9s} {len(df):7d} {df['quenched'].mean():6.3f}")

if not data:
    raise FileNotFoundError("no _allsat_ catalogs found -- run Part 1 first.")

## Helpers — anisotropy, bootstrap $f_q$, MCMC sinusoid fit (same as nb 06/14)

In [ ]:
def norm_hist(angles):
    c, _ = np.histogram(angles, bins=ANGLE_EDGES)
    return c / c.sum() / (90.0 / N_BINS)

def bootstrap_fq(angle, quenched, N=10000, seed=0):
    rng = np.random.default_rng(seed)
    angle = np.asarray(angle); quenched = np.asarray(quenched, dtype=float)
    n = len(angle)
    boot = np.full((N, N_BINS), np.nan)
    bin_idx = np.digitize(angle, ANGLE_EDGES) - 1
    for i in range(N):
        s = rng.integers(0, n, n)
        bi, qi = bin_idx[s], quenched[s]
        for j in range(N_BINS):
            m = bi == j
            if m.any():
                boot[i, j] = qi[m].mean()
    return np.nanmean(boot, axis=0), np.nanstd(boot, axis=0)

def log_likelihood(theta, x, y, sigma):
    a, b, f = theta
    s = sigma ** 2 + np.exp(f) ** 2
    model = a + b * np.cos(2 * np.radians(x))
    return -0.5 * np.sum((y - model) ** 2 / s + np.log(2 * np.pi * s))

def log_prior(theta):
    a, b, f = theta
    return 0.0 if (0 < a < 1 and -1 < b < 1 and -10 < f < 2) else -np.inf

def log_prob(theta, x, y, sigma):
    lp = log_prior(theta)
    return lp + log_likelihood(theta, x, y, sigma) if np.isfinite(lp) else -np.inf

def fit_sinusoid(mean, std, n_walkers=20, n_steps=10000, burn=1000, seed=0):
    np.random.seed(seed)
    ok = np.isfinite(mean) & np.isfinite(std) & (std > 0)
    p0 = np.array([0.7, 0.025, -3.0]) + 1e-2 * np.random.randn(n_walkers, 3)
    sampler = emcee.EnsembleSampler(n_walkers, 3, log_prob,
                                    args=(ANGLE_CENTERS[ok], mean[ok], std[ok]))
    sampler.run_mcmc(p0, n_steps, progress=False)
    chain = sampler.get_chain(discard=burn, flat=True)
    return chain.mean(axis=0), chain.std(axis=0)

def model_band(p, e, n_mc=10000, seed=0):
    rng = np.random.default_rng(seed)
    x = np.linspace(0, np.pi / 2, 400)
    a = rng.normal(p[0], e[0], n_mc); b = rng.normal(p[1], e[1], n_mc)
    y = a[:, None] + b[:, None] * np.cos(2 * x)[None, :]
    return np.degrees(x), (a.mean() + b.mean() * np.cos(2 * x)), np.percentile(y, 16, 0), np.percentile(y, 84, 0)

## Fit every (host cut × redshift × radius variant)

In [ ]:
hist, fq, params = {}, {}, {}
for key, df in data.items():
    hist[key]   = norm_hist(df["alpha"].to_numpy())
    fq[key]     = bootstrap_fq(df["alpha"].to_numpy(), df["quenched"].to_numpy())
    params[key] = fit_sinusoid(*fq[key])

print(f"{'host':<10s} {'redshift':<8s} {'variant':<9s}  {'a':>16s}   {'b':>16s}   |b|/sig_b")
for (hlabel, zlabel, vlabel), (p, e) in params.items():
    print(f"{hlabel:<10s} {zlabel:<8s} {vlabel:<9s}  {p[0]:.3f} +/- {e[0]:.3f}   "
          f"{p[1]:+.3f} +/- {e[1]:.3f}   {abs(p[1] / e[1]):.2f}")

## Two-panel plotter — overlay the radius variants

Left = anisotropy $P(\theta)$; right = $f_q(\theta)$ + sinusoid fit. Blue/solid = $<1\,R_{200c}$,
red/dashed = all satellites.

In [ ]:
_vstyle = {v[0]: v for v in VARIANTS}

def plot_radius_compare(hlabel, zlabel):
    keys = [(hlabel, zlabel, v[0]) for v in VARIANTS if (hlabel, zlabel, v[0]) in data]
    if not keys:
        print(f"({hlabel}, {zlabel}: missing)"); return
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 5))
    for key in keys:
        vlabel = key[2]
        _, color, ls, marker = _vstyle[vlabel]
        n = len(data[key])
        h = hist[key]
        axL.step(ANGLE_EDGES, np.r_[h, h[-1]], where="post", color=color, ls=ls, lw=2,
                 label=f"{vlabel} (N={n})")
        m, s = fq[key]
        axR.errorbar(ANGLE_CENTERS, m, yerr=s, fmt=marker, color=color, capsize=3, ms=5,
                     ls="none", label=vlabel)
        xdeg, ymed, ylo, yhi = model_band(*params[key])
        axR.plot(xdeg, ymed, color=color, ls=ls, lw=2)
        axR.fill_between(xdeg, ylo, yhi, color=color, alpha=0.12)
    for ax in (axL, axR):
        ax.set_xlim(0, 90); ax.set_xlabel(r"$\theta$ [deg]")
        ax.tick_params(which="both", direction="in", top=True, right=True)
    axL.set_ylim(0, 0.030); axL.set_ylabel(r"$P(\theta)$")
    axR.set_ylim(0, 1);     axR.set_ylabel(r"$f_q$")
    axL.set_title("Anisotropy"); axR.set_title("Quench fraction + MCMC fit")
    axL.legend(title="radius cut:", fancybox=False, edgecolor="k")
    fig.suptitle(f"TNG100, host {hlabel}, {zlabel}  ($M_{{*,\\rm sat}}>10^8\\,M_\\odot$)",
                 y=1.02, fontsize=14)
    plt.subplots_adjust(wspace=0.22)
    plt.show()

## 1. Radius comparison per host-mass cut and redshift

In [ ]:
for hlabel, *_ in HOST_CUTS:
    for zlabel, _ in REDS:
        plot_radius_compare(hlabel, zlabel)

## 2. Amplitude $b$: $<1\,R_{200c}$ vs all satellites

One panel per redshift: the fitted sinusoid amplitude $b$ (MCMC $1\sigma$ error) for each host-mass
cut, the two radius variants side by side. If the points overlap, the radius cut does not change the
inferred quenching anisotropy; a systematic offset means it does. Dashed line = isotropy ($b=0$).

In [ ]:
present = [h for h, *_ in HOST_CUTS if any((h, z[0], v[0]) in params for z in REDS for v in VARIANTS)]
x = np.arange(len(present))

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, (zlabel, _) in zip(axes, REDS):
    for (vlabel, color, ls, marker), dx in zip(VARIANTS, (-0.10, 0.10)):
        bs, es, xs = [], [], []
        for i, hlabel in enumerate(present):
            key = (hlabel, zlabel, vlabel)
            if key in params:
                p, e = params[key]
                bs.append(p[1]); es.append(e[1]); xs.append(i + dx)
        ax.errorbar(xs, bs, yerr=es, fmt=marker, color=color, capsize=4, ms=7, lw=2, label=vlabel)
    ax.axhline(0.0, color="grey", ls="--", lw=1)
    ax.set_xticks(x); ax.set_xticklabels(present)
    ax.set_xlabel(r"host-mass cut  [$\log_{10} M_{200c}$]")
    ax.set_title(f"TNG100, {zlabel}")
    ax.tick_params(which="both", direction="in", top=True, right=True)
    ax.legend(title="radius cut:", fancybox=False, edgecolor="k")
axes[0].set_ylabel(r"sinusoid amplitude  $b$")
fig.suptitle(r"Quenching-anisotropy amplitude: within $1\,R_{200c}$ vs all satellites "
             r"($M_{*,\rm sat}>10^8\,M_\odot$)", y=1.03, fontsize=14)
plt.subplots_adjust(wspace=0.05)
plt.show()

# companion table: amplitude shift from cutting at R200c
print(f"{'host':<10s} {'redshift':<8s} {'b(<R200c)':>11s} {'b(all)':>10s} {'Delta b':>9s}")
for hlabel in present:
    for zlabel, _ in REDS:
        ki, ka = (hlabel, zlabel, "<1 R200c"), (hlabel, zlabel, "all sat")
        if ki in params and ka in params:
            bi, ba = params[ki][0][1], params[ka][0][1]
            print(f"{hlabel:<10s} {zlabel:<8s} {bi:>+11.4f} {ba:>+10.4f} {ba - bi:>+9.4f}")